# 01 · Assemble the platform
*Hill requisitions the base.*

`azd up` already created the workspace, storage, and access connector. This notebook creates the
**Unity Catalog** objects on top: external location, catalog, schemas, and volumes.

**Before running**, create the storage credential `shield_cred` (see `docs/01-assemble-the-platform.md`).
You need the `CREATE EXTERNAL LOCATION` and `CREATE CATALOG` privileges (metastore admin has both).

In [ ]:
dbutils.widgets.text("storage_account", "", "Storage account name (azd output STORAGE_ACCOUNT_NAME)")
dbutils.widgets.text("credential", "shield_cred", "Storage credential name")
sa = dbutils.widgets.get("storage_account")
cred = dbutils.widgets.get("credential")
assert sa, "Fill in the storage_account widget"

## External location → catalog

In [ ]:
spark.sql(
    f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS shield_managed
    URL 'abfss://uc-managed@{sa}.dfs.core.windows.net/'
    WITH (STORAGE CREDENTIAL `{cred}`)
    """
)
spark.sql(
    f"""
    CREATE CATALOG IF NOT EXISTS shield
    MANAGED LOCATION 'abfss://uc-managed@{sa}.dfs.core.windows.net/shield'
    COMMENT 'S.H.I.E.L.D. intelligence platform'
    """
)

## Schemas = the medallion layers + landing + ops

In [ ]:
for schema in ["landing", "bronze", "silver", "gold", "ops"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS shield.{schema}")

spark.sql("CREATE VOLUME IF NOT EXISTS shield.landing.staging COMMENT 'Generated seed data'")
spark.sql("CREATE VOLUME IF NOT EXISTS shield.landing.raw COMMENT 'Files arriving from the field'")
spark.sql("CREATE VOLUME IF NOT EXISTS shield.landing.checkpoints COMMENT 'Streaming checkpoints and schemas'")

In [ ]:
display(spark.sql("SHOW SCHEMAS IN shield"))

In [ ]:
display(spark.sql("SHOW VOLUMES IN shield.landing"))

## Try it yourself
1. Create an *external* volume on the `landing` container (second external location) and compare it with the managed volume.
2. Run `DESCRIBE CATALOG EXTENDED shield` and find the managed location.
3. Add a `sandbox` schema and grant a colleague `USE SCHEMA` + `CREATE TABLE` on it only.